In [22]:
import time
import math
import io
import requests
import numpy as np
import pandas as pd
import pymysql
import FinanceDataReader as fdr
from datetime import datetime
from typing import Optional, Dict, Any, List

from DATA.stock_invest_function import get_db_host

# =========================================================
# 0) 설정
# =========================================================
DB_NAME = "investar"
TABLE_RESULT = "us_required_return_result"
PORT = 3307

MARKET_TICKER = "SPY"  # old_version notebook도 SPY 사용
MAX_RETRY = 5
SLEEP_BETWEEN_CALLS = 0.25

BETA_WINDOWS = [252, 750, 1250]  # beta_252, beta_750, beta_1250
START_LOOKBACK_BDAYS_IF_EMPTY = 252 * 6  # DB에 price 없으면 최소 6년만 (beta_1250 + 여유)

# =========================================================
# 1) DB 연결 / 조회
# =========================================================
def get_conn(db_info: Dict[str, Any]):
    return pymysql.connect(
        host=db_info["host"],
        port=db_info.get("port", PORT),
        user=db_info["user"],
        password=db_info["password"],
        db=db_info.get("database", DB_NAME),
        charset="utf8mb4",
        autocommit=False,
        cursorclass=pymysql.cursors.DictCursor
    )

def get_last_date_in_db(db_info: Dict[str, Any], ticker: str, indicator: str) -> Optional[pd.Timestamp]:
    sql = f"""
    SELECT MAX(date) AS last_date
    FROM {TABLE_RESULT}
    WHERE ticker=%s AND indicator=%s;
    """
    conn = get_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(sql, (ticker, indicator))
            row = cur.fetchone()
            last_dt = row["last_date"] if row else None
    finally:
        conn.close()
    return pd.to_datetime(last_dt) if last_dt is not None else None

def read_indicator_series(db_info: Dict[str, Any], ticker: str, indicator: str) -> pd.DataFrame:
    """
    returns: columns [date, value]
    """
    sql = f"""
    SELECT date, value
    FROM {TABLE_RESULT}
    WHERE ticker=%s AND indicator=%s
    ORDER BY date;
    """
    conn = get_conn(db_info)
    try:
        df = pd.read_sql(sql, conn, params=[ticker, indicator])
    finally:
        conn.close()

    if df.empty:
        return pd.DataFrame(columns=["date", "value"])

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.drop_duplicates(subset=["date"]).sort_values("date")
    return df

# =========================================================
# 2) FMP 호출
# =========================================================
def _get_json(url: str, params: Dict[str, Any]) -> Any:
    last_err = None
    for k in range(MAX_RETRY):
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code == 429:
                time.sleep(1.0 + 0.5 * k)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(0.5 + 0.5 * k)
    raise RuntimeError(f"FMP request failed after retries. last_err={last_err}")

def fetch_fmp_price(symbol: str, api_key: str, start_date: str) -> pd.DataFrame:
    """
    return columns [date, price] (adjClose 우선)
    """
    url = f"https://financialmodelingprep.com/api/v3/historical-price-full/{symbol}"
    params = {"from": start_date, "apikey": api_key}

    js = _get_json(url, params=params)
    hist = js.get("historical", []) if isinstance(js, dict) else []
    if not hist:
        return pd.DataFrame(columns=["date", "price"])

    df = pd.DataFrame(hist)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    price_col = "adjClose" if "adjClose" in df.columns else "close"
    df[price_col] = pd.to_numeric(df[price_col], errors="coerce")
    df = df[["date", price_col]].rename(columns={price_col: "price"})
    df = df.dropna(subset=["price"]).drop_duplicates("date").sort_values("date")
    return df

def fetch_us_treasury_yields(start_date: str, end_date: Optional[str] = None) -> pd.DataFrame:
    """
    old_version 방식 유지:
    1) FDR(FRED:DGS1,3,5) 시도
    2) 실패 시 Yahoo (^IRX, ^FVX) fallback + 3Y는 선형보간
    return columns: [rf_1y, rf_3y, rf_5y] in decimals
    """
    if end_date is None:
        end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

    # 1) FRED via FDR
    try:
        y1 = fdr.DataReader("FRED:DGS1", start_date, end_date)
        y3 = fdr.DataReader("FRED:DGS3", start_date, end_date)
        y5 = fdr.DataReader("FRED:DGS5", start_date, end_date)

        idx = y1.index.union(y3.index).union(y5.index)
        out = pd.DataFrame(index=idx).sort_index()
        out["rf_1y"] = pd.to_numeric(y1.iloc[:, 0], errors="coerce") / 100.0
        out["rf_3y"] = pd.to_numeric(y3.iloc[:, 0], errors="coerce") / 100.0
        out["rf_5y"] = pd.to_numeric(y5.iloc[:, 0], errors="coerce") / 100.0
        return out
    except Exception:
        pass

    # 2) Yahoo fallback
    y1 = fdr.DataReader("^IRX", start_date, end_date)  # 13-week proxy
    y5 = fdr.DataReader("^FVX", start_date, end_date)  # 5Y

    idx = y1.index.union(y5.index)
    out = pd.DataFrame(index=idx).sort_index()
    out["rf_1y"] = pd.to_numeric(y1["Close"], errors="coerce") / 100.0
    out["rf_5y"] = pd.to_numeric(y5["Close"], errors="coerce") / 100.0
    out["rf_3y"] = out["rf_1y"] + (out["rf_5y"] - out["rf_1y"]) * (3 - 1) / (5 - 1)
    return out

# =========================================================
# 3) 계산 (old_version 방식 유지: E_Rm = rolling mean * 252)
# =========================================================
def rolling_beta(ret_stock: pd.Series, ret_mkt: pd.Series, window: int) -> pd.Series:
    cov = ret_stock.rolling(window).cov(ret_mkt)
    var = ret_mkt.rolling(window).var()
    return cov / var

def build_features(price_stock_df: pd.DataFrame, price_mkt_df: pd.DataFrame, rf_df: pd.DataFrame) -> pd.DataFrame:
    """
    price_stock_df: [date, price_stock]
    price_mkt_df:   [date, price_mkt]
    rf_df index=date with [rf_1y, rf_3y, rf_5y]
    """
    df = pd.merge(price_stock_df, price_mkt_df, on="date", how="inner")
    df = df.sort_values("date").drop_duplicates("date")

    df["price_stock"] = pd.to_numeric(df["price_stock"], errors="coerce")
    df["price_mkt"]   = pd.to_numeric(df["price_mkt"], errors="coerce")
    df = df.dropna(subset=["price_stock", "price_mkt"])

    df["ret_stock"] = df["price_stock"].pct_change()
    df["ret_mkt"]   = df["price_mkt"].pct_change()

    # beta_252/750/1250
    for w in BETA_WINDOWS:
        df[f"beta_{w}"] = rolling_beta(df["ret_stock"], df["ret_mkt"], w)

    # rf merge (ffill)
    if rf_df is None or rf_df.empty:
        df["rf_1y"] = np.nan
        df["rf_3y"] = np.nan
        df["rf_5y"] = np.nan
    else:
        rf2 = rf_df.copy()
        rf2 = rf2.sort_index()
        rf2 = rf2.reindex(pd.to_datetime(df["date"])).ffill()
        df["rf_1y"] = rf2["rf_1y"].values
        df["rf_3y"] = rf2["rf_3y"].values
        df["rf_5y"] = rf2["rf_5y"].values

    # E_Rm (old_version: mean * 252)
    df["E_Rm_1y"] = df["ret_mkt"].rolling(252).mean() * 252
    df["E_Rm_3y"] = df["ret_mkt"].rolling(750).mean() * 252
    df["E_Rm_5y"] = df["ret_mkt"].rolling(1250).mean() * 252

    # Required return (각 horizon에 맞는 beta 사용: old_version 유지)
    df["Re_1y"] = df["rf_1y"] + df["beta_252"]  * (df["E_Rm_1y"] - df["rf_1y"])
    df["Re_3y"] = df["rf_3y"] + df["beta_750"]  * (df["E_Rm_3y"] - df["rf_3y"])
    df["Re_5y"] = df["rf_5y"] + df["beta_1250"] * (df["E_Rm_5y"] - df["rf_5y"])

    return df

# =========================================================
# 4) MySQL 저장: NaN/inf 절대 금지 + updated_at 없음 + PK 기반 중복 방지
# =========================================================
def _to_mysql_float(x: Any) -> Optional[float]:
    if x is None:
        return None
    try:
        v = float(x)
    except Exception:
        return None
    if math.isnan(v) or math.isinf(v):
        return None
    return v

def sanitize_long_for_mysql(df: pd.DataFrame) -> pd.DataFrame:
    """
    df columns = [date, ticker, indicator, value]
    """
    out = df.copy()

    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out = out[out["date"].notna()]
    out["date"] = out["date"].dt.date

    out["ticker"] = out["ticker"].astype(str)
    out["indicator"] = out["indicator"].astype(str)

    # ticker/indicator 오염 제거
    out = out[(out["ticker"].str.lower() != "nan") & (out["ticker"].str.lower() != "none")]
    out = out[(out["indicator"].str.lower() != "nan") & (out["indicator"].str.lower() != "none")]

    out["value"] = out["value"].apply(_to_mysql_float)

    return out

def upsert_long_df(
    db_info: Dict[str, Any],
    long_df: pd.DataFrame,
    batch_size_rows: int = 50_000,
    batch_size_ticker: int = 30,
    drop_null_values: bool = False
) -> None:
    """
    테이블에 updated_at 없다고 가정.
    PK(date,ticker,indicator)로 중복 방지.
    """
    if long_df is None or long_df.empty:
        return

    df = sanitize_long_for_mysql(long_df)

    if drop_null_values:
        df = df.dropna(subset=["value"])

    if df.empty:
        return

    tickers = sorted(df["ticker"].unique())
    n = len(tickers)
    print(f"[INFO] upsert 대상 ticker={n}, rows={len(df):,}")

    insert_sql = f"""
    INSERT INTO {TABLE_RESULT} (date, ticker, indicator, value)
    VALUES (%s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        value = VALUES(value);
    """

    for i in range(0, n, batch_size_ticker):
        batch_tickers = tickers[i:i+batch_size_ticker]
        batch = df[df["ticker"].isin(batch_tickers)].copy()
        batch = batch.sort_values(["ticker", "date", "indicator"])

        rows = list(batch[["date", "ticker", "indicator", "value"]].itertuples(index=False, name=None))

        conn = get_conn(db_info)
        try:
            with conn.cursor() as cur:
                for j in range(0, len(rows), batch_size_rows):
                    chunk = rows[j:j+batch_size_rows]

                    # 최종 방어: NaN/inf가 남아있으면 여기서 즉시 차단
                    for _r in chunk[:10]:
                        vv = _r[3]
                        if isinstance(vv, float) and (math.isnan(vv) or math.isinf(vv)):
                            raise ValueError(f"Still has NaN/inf in chunk sample: {_r}")

                    cur.executemany(insert_sql, chunk)
                    conn.commit()
            print(f"[OK] batch {i//batch_size_ticker+1}: tickers={len(batch_tickers)}, rows={len(rows):,}")
        except Exception as e:
            conn.rollback()
            raise
        finally:
            conn.close()

# =========================================================
# 5) 핵심 파이프라인: DB price_stock → 부족분만 FMP → 계산 → 저장
# =========================================================
def _bday_lookback_start(today_iso: str, bdays: int) -> str:
    start = (pd.to_datetime(today_iso) - pd.tseries.offsets.BDay(bdays)).date().isoformat()
    return start

def ensure_price_series(
    db_info: Dict[str, Any],
    ticker: str,
    api_key: str,
    indicator_name: str = "price_stock",
    today_iso: Optional[str] = None
) -> pd.DataFrame:
    """
    DB에서 (ticker, price_stock) 읽고,
    없거나/최신이 아니면 FMP로 부족분만 가져와 붙인 후
    '새로 받은 price 부분'은 DB에 upsert 저장까지 수행.
    return: columns [date, price_stock]
    """
    if today_iso is None:
        today_iso = datetime.utcnow().date().isoformat()

    # DB price 읽기
    df_db = read_indicator_series(db_info, ticker, indicator_name)  # [date,value]
    df_db = df_db.rename(columns={"value": indicator_name})

    last_dt = get_last_date_in_db(db_info, ticker, indicator_name)

    if last_dt is None:
        # DB에 price 자체가 없으면: 최소 6년만 받아서 채움 (용량 절약)
        start_date = _bday_lookback_start(today_iso, START_LOOKBACK_BDAYS_IF_EMPTY)
        df_new = fetch_fmp_price(ticker, api_key, start_date=start_date).rename(columns={"price": indicator_name})
        time.sleep(SLEEP_BETWEEN_CALLS)

        df_full = df_new.copy()
        # 받은 price는 DB에 저장
        if not df_new.empty:
            long_new = df_new.assign(ticker=ticker).melt(
                id_vars=["date", "ticker"], value_vars=[indicator_name],
                var_name="indicator", value_name="value"
            )
            upsert_long_df(db_info, long_new, drop_null_values=True)
        return df_full

    # last_dt 이후만 FMP로
    start_missing = (last_dt + pd.Timedelta(days=1)).date().isoformat()
    if start_missing <= today_iso:
        df_new = fetch_fmp_price(ticker, api_key, start_date=start_missing).rename(columns={"price": indicator_name})
        time.sleep(SLEEP_BETWEEN_CALLS)
    else:
        df_new = pd.DataFrame(columns=["date", indicator_name])

    df_full = pd.concat([df_db, df_new], ignore_index=True)
    df_full["date"] = pd.to_datetime(df_full["date"], errors="coerce")
    df_full = df_full.dropna(subset=["date"])
    df_full = df_full.drop_duplicates("date").sort_values("date")

    # 새로 받은 구간만 DB에 저장
    if not df_new.empty:
        long_new = df_new.assign(ticker=ticker).melt(
            id_vars=["date", "ticker"], value_vars=[indicator_name],
            var_name="indicator", value_name="value"
        )
        upsert_long_df(db_info, long_new, drop_null_values=True)

    return df_full

def update_one_ticker(
    db_info: Dict[str, Any],
    ticker: str,
    api_key: str,
    today_iso: Optional[str] = None,
    store_prices_and_returns: bool = True
) -> None:
    """
    - 종목 price_stock: DB 우선 + 부족분만 FMP
    - 시장 price_mkt: SPY를 DB에 ticker='SPY', indicator='price_stock'로 동일하게 관리(부족분만 FMP)
    - 계산 결과 + (원하면) price/return도 같이 저장
    """
    if today_iso is None:
        today_iso = datetime.utcnow().date().isoformat()

    # 1) 종목 가격 확보
    px_stock = ensure_price_series(db_info, ticker, api_key, indicator_name="price_stock", today_iso=today_iso)
    if px_stock.empty:
        print(f"[WARN] {ticker}: price_stock 비어있음. skip")
        return

    # 2) 시장(SPY) 가격 확보 (DB에 SPY price_stock로 저장/업데이트)
    px_spy = ensure_price_series(db_info, MARKET_TICKER, api_key, indicator_name="price_stock", today_iso=today_iso)
    if px_spy.empty:
        print(f"[WARN] {ticker}: SPY price 없음. skip")
        return

    # 3) merge용 컬럼명 변경 (시장 price를 price_mkt로)
    price_stock_df = px_stock.rename(columns={"price_stock": "price_stock"})
    price_mkt_df   = px_spy.rename(columns={"price_stock": "price_mkt"})

    # 4) rf
    start_for_rf = price_mkt_df["date"].min().date().isoformat()
    end_for_rf   = price_mkt_df["date"].max().date().isoformat()
    rf_df = fetch_us_treasury_yields(start_for_rf, end_for_rf)

    # 5) feature 계산 (old_version 방식)
    feat = build_features(price_stock_df, price_mkt_df, rf_df)
    feat["ticker"] = ticker

    # 6) 저장할 지표 선택
    base_cols = [
        "beta_252","beta_750","beta_1250",
        "rf_1y","rf_3y","rf_5y",
        "E_Rm_1y","E_Rm_3y","E_Rm_5y",
        "Re_1y","Re_3y","Re_5y",
    ]
    if store_prices_and_returns:
        base_cols = ["price_stock","price_mkt","ret_stock","ret_mkt"] + base_cols

    keep_cols = [c for c in base_cols if c in feat.columns]

    long_df = feat[["date","ticker"] + keep_cols].melt(
        id_vars=["date","ticker"],
        value_vars=keep_cols,
        var_name="indicator",
        value_name="value"
    )

    # 7) upsert (NaN/inf -> NULL 보장)
    upsert_long_df(db_info, long_df, drop_null_values=False)

    print(f"[OK] {ticker}: rows={len(long_df):,} upsert 완료")

def run_batch(
    db_info: Dict[str, Any],
    tickers: List[str],
    api_key: str,
    start_idx: int = 0,
    store_prices_and_returns: bool = True
) -> None:
    today_iso = datetime.utcnow().date().isoformat()

    # SPY는 배치 시작 전에 1번만 업데이트해두면 훨씬 효율적
    try:
        ensure_price_series(db_info, MARKET_TICKER, api_key, indicator_name="price_stock", today_iso=today_iso)
    except Exception as e:
        print(f"[ERROR] SPY 선업데이트 실패: {e}")
        return

    for i, t in enumerate(tickers[start_idx:], start=start_idx):
        try:
            update_one_ticker(
                db_info=db_info,
                ticker=t,
                api_key=api_key,
                today_iso=today_iso,
                store_prices_and_returns=store_prices_and_returns
            )
        except Exception as e:
            print(f"[ERROR] {t} 실패: {e}")


In [23]:
from DATA.stock_invest_function import get_db_host

# 1) DB 설정
db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",   # 실제 비밀번호
    "database": "investar",
}

API_KEY = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

tickers  = ["AAPL", "MSFT", "NVDA"]  # 예시

run_batch(
    db_info=db_info,
    tickers=tickers,
    api_key=API_KEY,
    start_idx=0,
    store_prices_and_returns=True
)

[INFO] upsert 대상 ticker=1, rows=1,456
[OK] batch 1: tickers=1, rows=1,456
[INFO] upsert 대상 ticker=1, rows=1,456
[OK] batch 1: tickers=1, rows=1,456
[WARN] AAPL: SPY price 없음. skip
[INFO] upsert 대상 ticker=1, rows=1,456
[OK] batch 1: tickers=1, rows=1,456
[WARN] MSFT: SPY price 없음. skip
[INFO] upsert 대상 ticker=1, rows=1,456
[OK] batch 1: tickers=1, rows=1,456
[WARN] NVDA: SPY price 없음. skip


In [14]:
# # 1) DB 설정
# db_info = {
#     "host": get_db_host(),
#     "port": 3307,
#     "user": "stox7412",
#     "password": "Apt106503!~",   # 실제 비밀번호
#     "database": "investar",
# }
#
# API_KEY = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'
#
# # TICKER_LIST = ["AAPL", "MSFT", "NVDA"]  # 예시
#
# TICKER_LIST = get_filtered_us_tickers()
# # TICKER_LIST = TICKER_LIST[1000:2000]
# TICKER_LIST = ["BG", "PLMR", "LLY", "AMD", "DASH", "LULU", "CAT"]

100%|██████████| 299/299 [00:00<00:00, 1138.13it/s]

제외된 기업 수: 1809
남은 기업 수: 4994
티커 수: 4994


In [15]:
# TICKER_LIST = TICKER_LIST[1500:2000]
# TICKER_LIST = ["APP"]

results: Dict[str, pd.DataFrame] = {}

for t in TICKER_LIST:
    try:
        df_t = run_capm_pipeline_for_ticker(
            ticker=t,
            api_key=API_KEY,
            market_symbol="^GSPC",     # S&P500 지수 기준
            start_date="2010-01-01"
        )
        results[t] = df_t
        print(f"[OK] {t} 처리 완료, {len(df_t)} rows")
        # FMP API 부담 줄이기 위해 살짝 대기
        time.sleep(1.0)
    except Exception as e:
        print(f"[ERROR] {t} 처리 중 오류: {e}")

[INFO] BG 가격 데이터 수집 (FMP)...
[INFO] ^GSPC 지수 데이터 수집 (FMP)...
[INFO] 일간 수익률 및 rolling 베타 계산...
[INFO] 미국 국채금리(1Y,3Y,5Y) 수집 (FDR/Yahoo fallback)...
[WARN] FDR 실패: Missing column provided to 'parse_dates': 'DATE'
[INFO] Yahoo Finance로 fallback합니다
[INFO] CAPM 필요수익률 계산...
[OK] BG 처리 완료, 4029 rows
[INFO] PLMR 가격 데이터 수집 (FMP)...
[INFO] ^GSPC 지수 데이터 수집 (FMP)...
[INFO] 일간 수익률 및 rolling 베타 계산...
[INFO] 미국 국채금리(1Y,3Y,5Y) 수집 (FDR/Yahoo fallback)...
[WARN] FDR 실패: Missing column provided to 'parse_dates': 'DATE'
[INFO] Yahoo Finance로 fallback합니다
[INFO] CAPM 필요수익률 계산...
[OK] PLMR 처리 완료, 1692 rows
[INFO] LLY 가격 데이터 수집 (FMP)...
[INFO] ^GSPC 지수 데이터 수집 (FMP)...
[INFO] 일간 수익률 및 rolling 베타 계산...
[INFO] 미국 국채금리(1Y,3Y,5Y) 수집 (FDR/Yahoo fallback)...
[WARN] FDR 실패: Missing column provided to 'parse_dates': 'DATE'
[INFO] Yahoo Finance로 fallback합니다
[INFO] CAPM 필요수익률 계산...
[OK] LLY 처리 완료, 4029 rows
[INFO] AMD 가격 데이터 수집 (FMP)...
[INFO] ^GSPC 지수 데이터 수집 (FMP)...
[INFO] 일간 수익률 및 rolling 베타 계산...
[INFO] 미국 국채금리(1Y,3Y

In [17]:
import pymysql

# 1) DB 설정
db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",   # 실제 비밀번호
    "database": "investar",
}

# 2) CAPM 파이프라인 돌려서 results 만들기 (이미 하셨음)
# results = { "AAPL": df_aapl, "MSFT": df_msft, ... }

# 3) results → long format (beta_*, Re_*만)
# long_df = results_to_long(results, indicator_prefixes=("beta_", "Re_"))

include_cols = [
    "price_stock", "price_mkt",
    "ret_stock", "ret_mkt",
    "beta_252", "beta_750", "beta_1250",
    "rf_1y", "rf_3y", "rf_5y",
    "E_Rm_1y", "E_Rm_3y", "E_Rm_5y",
    "Re_1y", "Re_3y", "Re_5y"
]
long_df = results_to_long_include(results, include_cols)


print(long_df.head())

# 4) 30개 티커씩 잘라서 DB에 저장
upsert_us_beta_re_long_df(
    long_df=long_df,
    db_info=db_info,
    table_name="us_required_return_result",
    batch_size_rows=50_000,
    batch_size_ticker=30
)

        date ticker    indicator  value
0 2010-01-05     BG  price_stock  66.88
1 2010-01-06     BG  price_stock  69.05
2 2010-01-07     BG  price_stock  70.93
3 2010-01-08     BG  price_stock  71.29
4 2010-01-11     BG  price_stock  70.28
[INFO] 테이블 확인/생성 완료: us_required_return_result
[INFO] 총 티커 수: 7, batch_size_ticker = 30
[INFO] 티커 batch 1: 7 tickers, 369808 rows 저장 예정
[ERROR] 티커 batch 저장 중 오류: nan can not be used with MySQL


ProgrammingError: nan can not be used with MySQL

In [8]:
results

{'BG':             price_stock  price_mkt  ret_stock   ret_mkt  beta_252  beta_750  \
 date                                                                          
 2010-01-05        66.88    1136.52   0.034493  0.003116       NaN       NaN   
 2010-01-06        69.05    1137.14   0.032446  0.000546       NaN       NaN   
 2010-01-07        70.93    1141.69   0.027227  0.004001       NaN       NaN   
 2010-01-08        71.29    1144.98   0.005075  0.002882       NaN       NaN   
 2010-01-11        70.28    1146.98  -0.014167  0.001747       NaN       NaN   
 ...                 ...        ...        ...       ...       ...       ...   
 2026-01-05        93.40    6902.04   0.008204  0.006351  0.415219  0.465635   
 2026-01-06        94.36    6944.83   0.010278  0.006200  0.416145  0.466612   
 2026-01-07        92.59    6920.92  -0.018758 -0.003443  0.421600  0.468456   
 2026-01-08        97.51    6921.45   0.053137  0.000077  0.419871  0.467166   
 2026-01-09        99.99    6966.2

In [22]:
results: Dict[str, pd.DataFrame] = {}

for t in TICKER_LIST:
    try:
        df_t = run_capm_pipeline_for_ticker(
            ticker=t,
            api_key=API_KEY,
            market_symbol="^GSPC",     # S&P500 지수 기준
            start_date="2010-01-01"
        )
        results[t] = df_t
        print(f"[OK] {t} 처리 완료, {len(df_t)} rows")
        # FMP API 부담 줄이기 위해 살짝 대기
        time.sleep(1.0)
    except Exception as e:
        print(f"[ERROR] {t} 처리 중 오류: {e}")

[INFO] NVDA 가격 데이터 수집 (FMP)...
[INFO] ^GSPC 지수 데이터 수집 (FMP)...
[INFO] 일간 수익률 및 rolling 베타 계산...
[INFO] 미국 국채금리(1Y,3Y,5Y) 수집 (FDR/Yahoo fallback)...
[WARN] FDR 실패: Missing column provided to 'parse_dates': 'DATE'
[INFO] Yahoo Finance로 fallback합니다
[INFO] CAPM 필요수익률 계산...
[OK] NVDA 처리 완료, 4000 rows
[INFO] AAPL 가격 데이터 수집 (FMP)...
[INFO] ^GSPC 지수 데이터 수집 (FMP)...
[INFO] 일간 수익률 및 rolling 베타 계산...
[INFO] 미국 국채금리(1Y,3Y,5Y) 수집 (FDR/Yahoo fallback)...
[WARN] FDR 실패: Missing column provided to 'parse_dates': 'DATE'
[INFO] Yahoo Finance로 fallback합니다
[INFO] CAPM 필요수익률 계산...
[OK] AAPL 처리 완료, 4000 rows
[INFO] MSFT 가격 데이터 수집 (FMP)...
[INFO] ^GSPC 지수 데이터 수집 (FMP)...
[INFO] 일간 수익률 및 rolling 베타 계산...
[INFO] 미국 국채금리(1Y,3Y,5Y) 수집 (FDR/Yahoo fallback)...
[WARN] FDR 실패: Missing column provided to 'parse_dates': 'DATE'
[INFO] Yahoo Finance로 fallback합니다
[INFO] CAPM 필요수익률 계산...
[OK] MSFT 처리 완료, 4000 rows
[INFO] AMZN 가격 데이터 수집 (FMP)...
[INFO] ^GSPC 지수 데이터 수집 (FMP)...
[INFO] 일간 수익률 및 rolling 베타 계산...
[INFO] 미국 국채금

In [23]:
df_all = pd.concat(results, names=["AVGO"])


In [24]:
df_all.tail(20)

price_stock  price_mkt  ret_stock   ret_mkt  beta_252  \
AVGO date                                                                
NFLX 2025-10-30       108.90    6822.35  -0.010360 -0.009905  0.907965   
     2025-10-31       111.89    6840.19   0.027456  0.002615  0.908966   
     2025-11-03       110.01    6851.98  -0.016802  0.001724  0.907785   
     2025-11-04       109.30    6771.54  -0.006454 -0.011740  0.917004   
     2025-11-05       109.85    6796.30   0.005032  0.003656  0.917497   
     2025-11-06       109.70    6720.31  -0.001365 -0.011181  0.914936   
     2025-11-07       110.37    6728.81   0.006108  0.001265  0.915378   
     2025-11-10       112.01    6832.42   0.014859  0.015398  0.917278   
     2025-11-11       113.64    6846.62   0.014552  0.002078  0.915227   
     2025-11-12       115.75    6850.93   0.018567  0.000630  0.915819   
     2025-11-13       115.42    6737.48  -0.002851 -0.016560  0.910274   
     2025-11-14       111.22    6734.10  -0.036389 -0.000502  0.913214   
     2025-11-17       110.29    6672.42  -0.008362 -0.009159  0.913563   
     2025-11-18       114.09    6617.33   0.034455 -0.008256  0.905770   
     2025-11-19       110.00    6642.15  -0.035849  0.003751  0.900194   
     2025-11-20       105.67    6538.77  -0.039364 -0.015564  0.909773   
     2025-11-21       104.31    6602.98  -0.012870  0.009820  0.901516   
     2025-11-24       106.97    6705.11   0.025501  0.015467  0.906345   
     2025-11-25       104.40    6765.89  -0.024025  0.009065  0.897253   
     2025-11-26       106.14    6812.60   0.016667  0.006904  0.899291   

                 beta_750  beta_1250    rf_1y     rf_3y    rf_5y   E_Rm_1y  \
AVGO date                                                                    
NFLX 2025-10-30  1.122784   1.288617  0.03757  0.037385  0.03720  0.176047   
     2025-10-31  1.122790   1.296852  0.03718  0.037175  0.03717  0.177048   
     2025-11-03  1.130791   1.297005  0.03783  0.037490  0.03715  0.182072   
     2025-11-04  1.132288   1.295684  0.03793  0.037470  0.03701  0.188947   
     2025-11-05  1.131534   1.296016  0.03788  0.037765  0.03765  0.188508   
     2025-11-06  1.126597   1.297353  0.03763  0.037260  0.03689  0.180142   
     2025-11-07  1.114149   1.299174  0.03757  0.037185  0.03680  0.169138   
     2025-11-10  1.108177   1.298943  0.03783  0.037465  0.03710  0.159243   
     2025-11-11  1.113979   1.300530  0.03783  0.037510  0.03719  0.153890   
     2025-11-12  1.111048   1.300523  0.03778  0.037230  0.03668  0.150763   
     2025-11-13  1.106794   1.299131  0.03793  0.037480  0.03703  0.133234   
     2025-11-14  1.105667   1.300518  0.03788  0.037615  0.03735  0.135622   
     2025-11-17  1.107360   1.301190  0.03772  0.037465  0.03721  0.126230   
     2025-11-18  1.101492   1.298513  0.03772  0.037330  0.03694  0.124027   
     2025-11-19  1.101657   1.297460  0.03772  0.037400  0.03708  0.140978   
     2025-11-20  1.106397   1.299810  0.03775  0.037240  0.03673  0.121499   
     2025-11-21  1.102880   1.297115  0.03740  0.036790  0.03618  0.127356   
     2025-11-24  1.104574   1.297697  0.03730  0.036675  0.03605  0.142797   
     2025-11-25  1.100427   1.295456  0.03732  0.036485  0.03565  0.146525   
     2025-11-26  1.078608   1.296374  0.03732  0.036485  0.03565  0.149961   

                  E_Rm_3y   E_Rm_5y     Re_1y     Re_3y     Re_5y  
AVGO date                                                          
NFLX 2025-10-30  0.212604  0.148410  0.163303  0.234118  0.180508  
     2025-10-31  0.217039  0.146579  0.164316  0.239125  0.179057  
     2025-11-03  0.213043  0.147209  0.168771  0.236003  0.179896  
     2025-11-04  0.205868  0.143299  0.176413  0.228145  0.174727  
     2025-11-05  0.205216  0.146048  0.176080  0.227241  0.178136  
     2025-11-06  0.208440  0.141050  0.168020  0.230111  0.172022  
     2025-11-07  0.190239  0.138957  0.158004  0.207710  0.169519  
     2025-11-10  0.192308  0.143027  0.149200  0.209058  0.